##### ### The University of Melbourne, School of Computing and Information Systems

# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID:** `1615557`


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **


## 1. Supervised model training


In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict as dd
from scipy.stats import norm

supervised_train_df = pd.read_csv("./datasets/adult_supervised_train.csv")
cleaned_df = supervised_train_df.dropna().copy()
cleaned_df["income"] = cleaned_df["income"].copy().map({"<=50K": 0, ">50K": 1})

classes = cleaned_df["income"].unique()
categorical_features = cleaned_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()
continuous_features = (
    cleaned_df.select_dtypes(include=[np.number])
    .drop(columns=["income"])
    .columns.tolist()
)

# Group by income, sorting into classes 0 and 1, and then find the mean and standard dev of each of the columns where those
# features are continuous. grouped_stats is a MultiIndex
grouped_stats = cleaned_df.groupby("income")[continuous_features].agg(["mean", "std"])
# This moves the feature from part of the column to part of the row, meaning we can now index (class_c, feature_x)
# and get mean and standard dev for that combination
stacked_stats = grouped_stats.stack(level=0, future_stack=True)
# Convert the MultiIndex cleanly into a dictionary with keys (class_c, feature_x) and value being dictionaries
# themselves, whose keys are "mean" and "std". Overall, it gives the mean and standard dev for the rows in the dataframe
# with class = c and feature = x
stats_dict = stacked_stats.to_dict("index")

# this is the dictionary of distributions, ready to be used on new unseen test data points
distribs_dict = dd(float)
for key in stats_dict:
    mean = stats_dict[key]["mean"]
    std = stats_dict[key]["std"]
    distribs_dict[key] = norm(loc=mean, scale=std)

# now must find P(x_j=v | c) for every combination of attribute, attribute value, and class
category_probabilities = dd(float)
unseen_probabilities = {}

# i guess i want the keys to be (class, attribute, attribute value) and the values to be the probabilities
ALPHA = 1
class_counts = {c: (cleaned_df["income"] == c).sum() for c in classes}
for cf in categorical_features:
    counts = pd.crosstab(
        cleaned_df["income"], cleaned_df[cf]
    )  # creates a 2d table of counts of how many times we have income = c and cf = v
    K_j = len(counts.columns)  # the number of distinct values for this feature
    for c in classes:
        unseen_probabilities[(c, cf)] = ALPHA / (class_counts[c] + ALPHA * K_j)
        for av in counts.columns:
            count = counts.loc[
                c, av
            ]  # means the row label is c and the column label is av. this gives the # of occurrences
            # where class is c and xj = av
            category_probabilities[(c, cf, av)] = (count + ALPHA) / (
                class_counts[c] + ALPHA * K_j
            )


def find_most_predictive_att_val(c, ft):
    """
    Finds the attribute value most predictive of a given class c, and returns that attribute and its
    R value as a dictionary.
    """
    highest_R_val = 0
    att = None
    for av in cleaned_df[ft].unique():
        p0 = category_probabilities[(0, ft, av)]  # chance of <=50K
        p1 = category_probabilities[(1, ft, av)]  # chance of >50K
        R_val = p1 / p0 if c == 1 else p0 / p1
        if R_val > highest_R_val:
            highest_R_val = R_val
            att = av
    return {"class": c, "attribute": att, "R_val": round(highest_R_val, 3)}


# --- Finding Most Predictive Attribute Value for Each Category ---
highest_R_val_per_ft = dd(dict)
for cf in categorical_features:
    for c in classes:
        highest_R_val_per_ft[cf][c] = find_most_predictive_att_val(c, cf)

# --- Printing Most Predictive Attribute Value for Each Category ---
print("------- Printing Most Predictive Attribute Value for Each Category -------\n")
for cf in categorical_features:
    for c in sorted(classes):
        print(f"-Category {cf} and Class {c}-")
        most_predictive_att_val = highest_R_val_per_ft[cf][c]["attribute"]
        corresponding_r_val = highest_R_val_per_ft[cf][c]["R_val"]
        print(
            f"Attribute: {most_predictive_att_val:<12} | R-Value: {corresponding_r_val}\n"
        )


# --- Printing Top 5 Predictive Attributes for Each Class ---
highest_R_val_per_ft_list = [(key, val) for key, val in highest_R_val_per_ft.items()]
for c in classes:
    print(f"\n----- Top 5 Predictive Attributes for Class {c} -----\n")
    sorted_list = sorted(
        highest_R_val_per_ft_list, key=lambda x: x[1][c]["R_val"], reverse=True
    )
    for feature_name, data in sorted_list[0:5]:
        attribute_name = data[c]["attribute"]
        r_val = data[c]["R_val"]
        print(
            f"Feature: {feature_name:<15} | Attribute: {attribute_name:<18} | R-Value: {r_val}"
        )

prior_below_50k = (cleaned_df["income"] == 0).sum() / len(cleaned_df)
prior_above_50k = (cleaned_df["income"] == 1).sum() / len(cleaned_df)
print("\n----- Printing Priors -----\n")
print("prior_above_50k", round(prior_above_50k, 3))
print("prior_below_50k", round(prior_below_50k, 3))
print("\n----- Printing Continuous Feature Means and Std Devs -----")
for key in sorted([key for key in stats_dict.keys()], key=lambda x: x[1]):
    print()
    print(f"Class: {key[0]}, feature: {key[1]}")
    print(f"Mean: {round(stats_dict[key]['mean'], 3)}")
    print(f"Standard dev: {round(stats_dict[key]['std'], 3)}")

------- Printing Most Predictive Attribute Value for Each Category -------

-Category workclass and Class 0-
Attribute: Private      | R-Value: 1.167

-Category workclass and Class 1-
Attribute: Self-emp-inc | R-Value: 3.251

-Category education and Class 0-
Attribute: 9th          | R-Value: 6.629

-Category education and Class 1-
Attribute: Prof-school  | R-Value: 7.959

-Category marital-status and Class 0-
Attribute: Never-married | R-Value: 6.104

-Category marital-status and Class 1-
Attribute: Married-AF-spouse | R-Value: 4.288

-Category occupation and Class 0-
Attribute: Priv-house-serv | R-Value: 11.932

-Category occupation and Class 1-
Attribute: Exec-managerial | R-Value: 2.768

-Category relationship and Class 0-
Attribute: Own-child    | R-Value: 19.168

-Category relationship and Class 1-
Attribute: Wife         | R-Value: 2.745

-Category race and Class 0-
Attribute: Amer-Indian-Eskimo | R-Value: 2.267

-Category race and Class 1-
Attribute: Asian-Pac-Islander | R-Valu

## 2. Supervised model evaluation


In [3]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import math

test_set_df = pd.read_csv("./datasets/adult_test.csv")
test_set_df = test_set_df.dropna().copy()
test_set_df["income"] = test_set_df["income"].map({"<=50K": 0, ">50K": 1}).copy()
print(cleaned_df["native-country"].unique())


def calculate_likelihood_continuous(cj, xj):
    """
    Calculates the likelihood of an instance having attribute j being of value x, given that its class is c.
    """
    distrib = distribs_dict[
        cj
    ]  # get the normal distrib scaled for that class-feature combo with the mean and variance calculated from the data
    likelihood = distrib.pdf(xj)
    return max(
        likelihood, 1e-15
    )  # return tiny epsilon to prevent taking the log of 0 in case the probability is tiny


def calculate_posterior(likelihoods_list, c):
    """
    Calculates the posterior probability of an instance being of class c.
    """
    prior = class_counts[c] / len(cleaned_df)
    # start with the log prior and add on the log likelihoods
    log_posterior = math.log(prior)
    for l in likelihoods_list:
        log_posterior += math.log(l)
    # log is monotonic, so the highest log value will also be the highest non-log value
    return log_posterior


unseen_cf_value_count = 0


def classify_instance(instance: pd.Series):
    """
    Returns the more likely class of a test instance, based on the values of its attributes.
    """
    global unseen_cf_value_count
    leakproof_instance = instance.drop(labels=["income"], errors="ignore")
    # because probabilities are between 0 and 1, log(probability) will never be >0. The closer the probability
    # to 1, the less contribution adding it will make to the log sum, so the bigger (i.e. the less negative)
    # the sum of the logs is, the bigger the probability.
    max_log_posterior = -float("inf")
    best_class = None
    posteriors_dict = {}
    for c in classes:
        likelihoods_list = []
        for ft in leakproof_instance.index:
            av = leakproof_instance[ft]
            if ft in continuous_features:
                cj = (c, ft)  # class, feature
                likelihood = calculate_likelihood_continuous(cj=cj, xj=av)
                likelihoods_list.append(likelihood)
            else:
                likelihood = category_probabilities.get((c, ft, av), None)
                if likelihood is None:
                    likelihood = unseen_probabilities[(c, ft)]
                    if c == classes[0]:
                        likelihood_in_other_class = category_probabilities.get(
                            (classes[1], ft, av), None
                        )
                        if (
                            likelihood_in_other_class is None
                        ):  # this way we're sure it appeared in neither class
                            unseen_cf_value_count += 1
                            print(ft, av)
                likelihoods_list.append(likelihood)
        posterior = calculate_posterior(likelihoods_list, c)
        posteriors_dict[c] = posterior
        if posterior > max_log_posterior:
            max_log_posterior = posterior
            best_class = c
    # high ratio here means the model is very confident it's 0 as opposed to 1, and low ratio means the model
    # is very confident it's 1 as opposed to 0
    log_posterior_diff = (
        posteriors_dict[0] - posteriors_dict[1]
    )  # still working in log space
    # if model is very confident, it's possible for numbers to overflow, so we catch that just in case
    try:
        posterior_ratio = math.exp(log_posterior_diff)
    except OverflowError:
        posterior_ratio = float("inf") if log_posterior_diff > 0 else 0.0
    return best_class, posterior_ratio


# the slow part. This takes like 10 sec
test_set_df[["prediction", "posterior_ratio"]] = test_set_df.apply(
    classify_instance, axis=1, result_type="expand"
)

accuracy = accuracy_score(test_set_df["income"], test_set_df["prediction"])
print("accuracy:", round(accuracy, 3), "\n")

print(classification_report(test_set_df["income"], test_set_df["prediction"]))

print("unseen_cf_value_count:", unseen_cf_value_count, "\n")

print(confusion_matrix(test_set_df["income"], test_set_df["prediction"]), "\n")

# --- Printing Records with High Confidence of Class 0 and Class 1
confident_of_c0_records = test_set_df.sort_values(
    by="posterior_ratio", ascending=False
).iloc[0:3]
confident_of_c1_records = test_set_df.sort_values(
    by="posterior_ratio", ascending=True
).iloc[0:3]
test_set_df["R_dist_from_1"] = abs(test_set_df["posterior_ratio"] - 1)
near_decision_boundary_records = test_set_df.sort_values(by="R_dist_from_1").iloc[0:3]
print(
    "records with highest confidence that income is <=50K:\n", confident_of_c0_records
)
print("records with highest confidence that income is >50K:\n", confident_of_c1_records)
print("records near decision boundary:\n", near_decision_boundary_records)

['United-States' 'Italy' 'Mexico' 'Scotland' 'China' 'Japan' 'Poland'
 'Germany' 'Dominican-Republic' 'Cuba' 'South' 'Greece' 'Honduras'
 'Jamaica' 'Nicaragua' 'El-Salvador' 'Ireland' 'Laos' 'Peru' 'England'
 'France' 'Philippines' 'Canada' 'India' 'Taiwan' 'Columbia' 'Vietnam'
 'Hungary' 'Portugal' 'Ecuador' 'Hong' 'Thailand' 'Puerto-Rico' 'Haiti'
 'Iran' 'Guatemala' 'Trinadad&Tobago' 'Yugoslavia' 'Cambodia'
 'Outlying-US(Guam-USVI-etc)']
native-country Holand-Netherlands
accuracy: 0.83 

              precision    recall  f1-score   support

           0       0.85      0.93      0.89     11303
           1       0.72      0.52      0.61      3784

    accuracy                           0.83     15087
   macro avg       0.79      0.73      0.75     15087
weighted avg       0.82      0.83      0.82     15087

unseen_cf_value_count: 1 

[[10541   762]
 [ 1804  1980]] 

records with highest confidence that income is <=50K:
        age workclass  fnlwgt education  education-num marital-s

## 3. Extending the model with semi-supervised training


## 4. Supervised model evaluation
